<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Full_Fine_Tuning_with_Unsloth_Example_with_Llama_3_1_8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*More details in this article: [Fast and Memory-Efficient Full Fine-Tuning with Unsloth (single-GPU)](https://kaitchup.substack.com/p/fast-and-memory-efficient-full-fine)*

This notebook shows how to fully fine-tune LLMs with Unsloth. It fine-tune Llama 3.1 8B (base model) to make a state-of-the-art chat model using TULU 3's tokenizer, hyperparameters, and dataset.

You will need a 48 GB GPU. If you decrease the maximum sequence length, it will require less memory. The training logs shown in this notebook have been generated with an NVIDIA RTX A6000.

# Check How to Install Unsloth for Your Configuration

The output is the command that you have to run to properly install Unsloth.

In [ ]:
!wget -qO- https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py | python -

pip install --upgrade pip && pip install "unsloth[cu124-ampere-torch260] @ git+https://github.com/unslothai/unsloth.git"


# Install Unsloth

In [ ]:
pip install --upgrade pip && pip install "unsloth[cu124-ampere-torch260] @ git+https://github.com/unslothai/unsloth.git"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ib74tg8r/unsloth_d1332f26ada64eae8d73090bb51d283a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ib74tg8r/unsloth_d1332f26ada64eae8d73090bb51d283a
  Resolved https://github.com/unslothai/unsloth.git to commit eefba34e94443971533bffdf2ac32069ed07b0c2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 95.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 75.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0

In [ ]:
from unsloth import FastLanguageModel
import torch, os, multiprocessing, gc
from datasets import load_dataset
from peft import LoraConfig
from transformers import set_seed, AutoTokenizer

from trl import SFTTrainer, SFTConfig

set_seed(42)

model_name = "meta-llama/Llama-3.1-8B"
tokenizer_name_chat_template = "allenai/Llama-3.1-Tulu-3-8B"
compute_dtype = torch.bfloat16

bs = 2 #Batch size per device (training and validation), bs = 1 *can* be faster
gas = 64 #Gradient accumulation steps
mseqlen = 4096 #Maximum sequence length; reduce if you run out of memory

lr = 5e-6 #Default learning rate, way too small for Unsloth. Multiply it by 10 or 20 .

output_dir = "./SFT/"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    fix_tokenizer=False,
    max_seq_length = mseqlen,
    dtype = compute_dtype,
    full_finetuning=True
)

ds_train = load_dataset("allenai/tulu-3-sft-mixture", split="train")

# Apply the chat template from TULU's tokenizer
tokenizer_chat = AutoTokenizer.from_pretrained(tokenizer_name_chat_template)
def process(row):
    row["text"] = tokenizer_chat.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)
    return row

ds_train = ds_train.map(
    process,
    num_proc= multiprocessing.cpu_count(),
    load_from_cache_file=False,
)

ds_train = ds_train.remove_columns(["messages"])

training_arguments = SFTConfig(
        output_dir=output_dir,
        #eval_strategy="steps",
        #do_eval=True,
        optim="paged_adamw_8bit",
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=gas,
        #per_device_eval_batch_size=bs,
        log_level="debug",
        save_strategy="steps",
        save_steps=10000,
        logging_steps=25,
        learning_rate=lr,
        bf16 = True,
        #eval_steps=25,
        num_train_epochs=2,
        warmup_ratio=0.03,
        report_to = "none",
        lr_scheduler_type="linear",
        max_seq_length=mseqlen,
        dataset_text_field='text',
        dataset_num_proc=multiprocessing.cpu_count()
)

trainer = SFTTrainer(
    model = model,
    train_dataset=ds_train,
    #eval_dataset=ds_test,
    processing_class=tokenizer,
    args = training_arguments
)

trainer_ = trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: You selected full finetuning support, but 4bit / 8bit is enabled - disabling LoRA / QLoRA.
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.3.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.529 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Map (num_proc=64):   0%|          | 0/939343 [00:00<?, ? examples/s]

Using auto half precision backend
Currently training with a batch size of: 2
The following columns in the training set don't have a corresponding argument in `LlamaForCausalLM.forward` and have been ignored: text, id, source. If text, id, source are not expected by `LlamaForCausalLM.forward`,  you can safely ignore this message.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 939,343 | Num Epochs = 2 | Total steps = 14,676
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 64
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 64 x 1) = 128
 "-____-"     Trainable parameters = 8,030,261,248/8,030,261,248 (100.00% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,1.219200
50,1.234600
75,1.179700
100,1.094700
125,1.115400
150,1.040900
175,1.055600
200,1.006900
225,1.029700
250,0.984600
